# Excel → TTL ingestion

Three steps:

1. **Copy the template.** Take `data/ingestion_template/data_ingestion_template.xlsx` and paste it into `data/ingestion/input/` under your own filename (e.g. `my_project.xlsx`). Or run the optional copy cell below.
2. **Fill it in.** Open your copy in Excel, replace the demo rows with your components, save. Keep the header rows on top of every sheet — the Excel template has data-validation dropdowns that match the importer's vocabulary.
3. **Convert.** In the **Convert** section below, pick your workbook from the dropdown and click *Convert to TTL*. The project URI defaults to `https://digicities.info/proj/<your_filename>` — edit if you want a different namespace.

Full reference for the workbook format: [`tutorial/09_excel_import.ipynb`](../../tutorial/09_excel_import.ipynb). A paste-ready cheatsheet for every attribute type: [`data/ingestion_template/`](../ingestion_template/).

## Setup

In [ ]:
import os, sys, pathlib

REPO_ROOT = pathlib.Path().resolve().parent.parent  # data/ingestion → repo root
sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault('GRAPHDB_URL', 'http://localhost:7201')

INPUT_DIR  = REPO_ROOT / 'data' / 'ingestion' / 'input'
OUTPUT_DIR = REPO_ROOT / 'data' / 'ingestion' / 'output'
TEMPLATE   = REPO_ROOT / 'data' / 'ingestion_template' / 'data_ingestion_template.xlsx'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

available = sorted(p.name for p in INPUT_DIR.glob('*.xlsx') if not p.name.startswith('~$'))
print(f'Workbooks in {INPUT_DIR.name}/:')
for f in available:
    print(f'  - {f}')

print()
print(f'Drop folder : {INPUT_DIR}')
print(f'Template    : {TEMPLATE}')

## (Optional) Copy the template under a new name

If you'd rather copy the template from Python than File Explorer, edit `NEW_FILENAME` and run this cell. Skip otherwise.

In [ ]:
import shutil

NEW_FILENAME = 'my_project.xlsx'   # <-- edit this to your project name

target = INPUT_DIR / NEW_FILENAME
if target.exists():
    print(f'{target.name} already exists in input/ — skipping copy.')
elif not TEMPLATE.exists():
    print(f'Template not found at {TEMPLATE} — check the path.')
else:
    shutil.copy(TEMPLATE, target)
    print(f'copied template → {target}')
    print('Now open it in Excel, fill in your data, save, then run the Convert cell below.')

## Convert your workbook to TTL

Pick the workbook from the dropdown. The project URI auto-populates to `https://digicities.info/proj/<your_filename>` and updates whenever you change the selection — override if you want a different namespace. Click **Convert to TTL**; the result lands in `data/ingestion/output/<your_workbook>.ttl`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import rdflib
from backend.replica_builder.utils.create_class_and_attribute_graph import process_excel_to_ttl

# Re-scan input/ each time the cell runs so newly-added workbooks show up.
available = sorted(p.name for p in INPUT_DIR.glob('*.xlsx') if not p.name.startswith('~$'))

if not available:
    print(f'No .xlsx files in {INPUT_DIR.name}/. Copy the template above and re-run this cell.')
else:
    _default_uri = lambda stem: f'https://digicities.info/proj/{stem}'

    file_dd = widgets.Dropdown(
        options=available,
        description='Workbook:',
        layout=widgets.Layout(width='60%'),
    )
    uri_txt = widgets.Text(
        value=_default_uri(pathlib.Path(available[0]).stem),
        description='Project URI:',
        layout=widgets.Layout(width='60%'),
    )
    btn = widgets.Button(description='Convert to TTL', button_style='primary')
    out = widgets.Output()

    def _sync_uri(change):
        uri_txt.value = _default_uri(pathlib.Path(change['new']).stem)
    file_dd.observe(_sync_uri, names='value')

    def _on_click(_):
        with out:
            clear_output()
            src = INPUT_DIR / file_dd.value
            dst = OUTPUT_DIR / (src.stem + '.ttl')
            print(f'Converting {src.name} …')
            try:
                process_excel_to_ttl(
                    project_uri=uri_txt.value,
                    file_path=str(src),
                    output_ttl_path=str(dst),
                    uri_mode='default',
                )
                g = rdflib.Graph()
                g.parse(dst, format='turtle')
                print(f'OK — wrote {len(g)} triples to {dst}')
                # Expose for the upload cell below.
                globals()['INPUT_FILE']  = file_dd.value
                globals()['PROJECT_URI'] = uri_txt.value
                globals()['dst']         = dst
            except Exception as e:
                print(f'ERROR: {e}')
    btn.on_click(_on_click)

    display(file_dd, uri_txt, btn, out)

## (Optional) Upload to your GraphDB workspace

Pushes the TTL into a named graph in your workspace repository. The named graph is your project URI — same workbook + same project URI = same graph, so re-running overwrites cleanly.

Run the **Convert** cell above first so this cell knows which file to upload. Skip this cell if you just want a TTL file on disk.

In [ ]:
from backend.graphdb import GraphDBClient

REPO_NAME  = os.environ.get('LOCAL_WORKSPACE', 'workspace_demo')
GRAPH_NAME = f'<{PROJECT_URI}>'

client = GraphDBClient(token='local', selected_repo=REPO_NAME)
if not client.test_connection():
    raise RuntimeError(
        f'Could not reach GraphDB at {os.environ["GRAPHDB_URL"]}. '
        f'Is the stack up? `docker compose up -d` from the repo root.'
    )

client.upload_ttl(
    ttl_str=dst.read_text(encoding='utf-8'),
    graph_name=GRAPH_NAME,
    replace_existing=True,
)
print(f'uploaded {dst.name} → {REPO_NAME} / {GRAPH_NAME}')